# IADrift development tuning (SR ×4, development images)

Follow-up to the first 4-image IADrift development run (`scripts/run_sr4_iadrift_dev.py`, θ lr 1e-5, ε lr 1e-3, 1,000 steps). That run showed:
- IADrift is much better than CSGM below the LR resolution limit.
- Its invented detail above the limit is worse than none, so HR PSNR fell from the CSGM start at every step.

This notebook tests gentler adaptation, back-projection and frequency fusion on the **same development images**. No test images are used.

## Setup

In [ ]:
import os
import sys
from pathlib import Path

os.environ["XLA_PYTHON_CLIENT_ALLOCATOR"] = "platform"  # must be set before JAX is imported
SCRIPTS = Path.home() / "work" / "IADrift" / "scripts"
if str(SCRIPTS) not in sys.path:
    sys.path.insert(0, str(SCRIPTS))

import io
import json

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, display
from jax.scipy.sparse.linalg import cg

# Tested helpers from the project scripts (this import loads the generator).
from freq_metrics import BANDS, LR_NYQUIST, band_snr_db, fuse_with_bicubic
from run_sr4_csgm_test import FROZEN_CONFIG, load_split
from run_sr4_iadrift_dev import check_dev, csgm_cached
from run_sr4_iadrift_smoke import (
    A_sr4, DATA_DIR, atomic_write_bytes, bicubic_up, class_name, eager_metrics,
    generate_jit, jax, jnp, load_hr256, load_index, make_joint_step, params, psnr,
)

BAND_NAMES = [name for name, *_ in BANDS]
%matplotlib inline
# (the imported scripts select the non-interactive Agg backend; switch back for inline figures)
print("Devices:", jax.devices())

## Back-projection (IAGAN's final step)

$x_{BP} = x + A^T (A A^T)^{-1} (y - A x)$: the image closest to $x$ (in L2) that reproduces the measurement $y$ exactly. $A^T$ comes from `jax.linear_transpose`, and $(AA^T)^{-1}$ is applied with conjugate gradients, so no matrix is ever formed. Each use is checked by the LR loss of $x_{BP}$, which should be close to 0.

In [ ]:
_A_transpose = jax.linear_transpose(A_sr4, jnp.zeros((1, 256, 256, 3), jnp.float32))


@jax.jit
def _backproject(x, y):
    def AT(v):
        return _A_transpose(v)[0]
    u, _ = cg(lambda v: A_sr4(AT(v)), y - A_sr4(x), tol=1e-6, maxiter=500)
    return x + AT(u)


def backproject(x, y):
    """x: [256, 256, 3] numpy; y: [1, 64, 64, 3]. Returns the back-projected image."""
    return np.asarray(_backproject(jnp.asarray(x, jnp.float32)[None], y)[0])


def lr_loss(x, y):
    return float(jnp.sum((A_sr4(jnp.asarray(x, jnp.float32)[None]) - y) ** 2))


def evaluate(x, hr, y):
    return {"psnr": psnr(hr, x), "lr_loss": lr_loss(x, y), "band_snr_db": band_snr_db(x, hr)}

## Experiment: what we did

**Question.** Can gentler adaptation, back-projection or frequency fusion keep IADrift's gains below the LR limit without the damage above it?

**Images.** 4 **development** images: 8117 (streetcar), 3939 (bonnet), 403 (crane), 347 (spoonbill).
- The same images as the first IADrift development run, drawn with seed 2030 from the development-sweep pool.
- `check_dev` refuses any image in the test split.

**Label.** Correct (oracle) class only.

**Starting point.** For each image, the cached CSGM solution: frozen generator, Adam lr 0.02, 5,000 steps, 2 restarts, keeping the restart with the lowest LR loss.

**IADrift.**
- Joint Adam on the generator weights θ (starting from the pretrained weights) and the input noise ε, minimizing $\|y - A f_\theta(\epsilon)\|^2$ for 1,000 steps.
- 6 settings: θ lr ∈ {1e-6, 3e-6, 1e-5} × ε lr ∈ {0 (ε frozen), 1e-3}.
- (1e-5, 1e-3) repeats the earlier run.

**During the run.** Every 50 steps (and at step 0) we record LR loss, HR PSNR and band SNR. This is evaluation only: HR is never used by the optimization or to stop a run.

**After 1,000 steps, three outputs per run:**
1. **IADrift**: the generator output.
2. **IADrift + BP**: back-projected onto $\{x : Ax = y\}$ (cell above).
3. **IADrift + fusion**: bicubic below 0.0625 and above 0.125 cycles/pixel, IADrift between 0.0625 and 0.125 (the mid band, just below the LR limit).

**Baselines on the same images:** bicubic, CSGM, CSGM + BP, CSGM + fusion.

**Metrics:** HR PSNR (dB), LR loss, and band SNR per frequency band (`freq_metrics.py`).
- Final images come from the eager generator path, like the saved CSGM numbers.
- The every-50-steps curves use the compiled path, so their values are slightly offset (see the numerics note in `Untitled-1.ipynb`).

**Caveats.**
- Only 4 images.
- The fusion band edges were chosen after looking at the earlier development results.
- The "best setting" below is selected on these same 4 images.

Results are cached in `~/work/datasets/ImageNetV2/iadrift_dev_tuning_v1/`; re-running the cell loads them instead of recomputing.

In [ ]:
DEV_IMAGES = [8117, 3939, 403, 347]
CONFIGS = [(lr_theta, lr_eps) for lr_theta in (1e-6, 3e-6, 1e-5) for lr_eps in (0.0, 1e-3)]
STEPS = 1_000
EVAL_EVERY = 50
OUT_DIR = DATA_DIR / "iadrift_dev_tuning_v1"

index_rows = load_index()
labels = np.array([int(row["label"]) for row in index_rows])
train_indices, test_indices = load_split(labels)
check_dev(DEV_IMAGES, train_indices, test_indices)  # refuses test images
lr_all = np.load(DATA_DIR / "imageNet_x4.npy", mmap_mode="r")

# Baselines from the cached CSGM solutions (correct label).
images = {}
for i in DEV_IMAGES:
    label = int(labels[i])
    hr = load_hr256(DATA_DIR / index_rows[i]["relative_path"])
    y = jnp.asarray(lr_all[i:i + 1], jnp.float32)
    restarts = [csgm_cached(i, "oracle", r, label, lr_all[i], hr) for r in range(FROZEN_CONFIG["num_restarts"])]
    best = min(range(len(restarts)), key=lambda r: (restarts[r][0]["final_loss"], r))
    bic, csgm = bicubic_up(lr_all[i]), restarts[best][1]["x_hat"]
    outputs = {"bicubic": bic, "CSGM": csgm, "CSGM + BP": backproject(csgm, y),
               "CSGM + fusion": fuse_with_bicubic(csgm, bic)}
    images[i] = {"label": label, "hr": hr, "y": y, "csgm": restarts[best][1], "outputs": outputs,
                 "metrics": {name: evaluate(x, hr, y) for name, x in outputs.items()}}


def run_iadrift(i, lr_theta, lr_eps, joint_step, opt_theta, opt_eps):
    """One IADrift run from the CSGM solution; cached as an .npz."""
    path = OUT_DIR / f"img{i:05d}_lrt{lr_theta:g}_lre{lr_eps:g}.npz"
    if path.exists():
        with np.load(path) as data:
            return json.loads(str(data["result_json"])), {k: data[k] for k in ("IADrift", "IADrift + BP", "IADrift + fusion")}

    im = images[i]
    theta = jax.device_put(params)  # fresh copy of the pretrained weights
    eps, noise_labels = jnp.asarray(im["csgm"]["eps"]), jnp.asarray(im["csgm"]["noise_labels"])
    state_theta, state_eps = opt_theta.init(theta), opt_eps.init(eps)
    trace = {"step": [], "psnr": [], "lr_loss": [], **{b: [] for b in BAND_NAMES}}

    def record(step):
        x = np.asarray(generate_jit(theta, eps, im["label"], noise_labels)[0])
        m = evaluate(x, im["hr"], im["y"])
        trace["step"].append(step)
        trace["psnr"].append(m["psnr"])
        trace["lr_loss"].append(m["lr_loss"])
        for b in BAND_NAMES:
            trace[b].append(m["band_snr_db"][b])

    record(0)
    for step in range(1, STEPS + 1):
        theta, eps, state_theta, state_eps, _ = joint_step(
            theta, eps, state_theta, state_eps, im["y"], im["label"], noise_labels)
        if step % EVAL_EVERY == 0:
            record(step)

    x, _, _ = eager_metrics(theta, eps, im["label"], noise_labels, im["y"], im["hr"])
    outputs = {"IADrift": x, "IADrift + BP": backproject(x, im["y"]),
               "IADrift + fusion": fuse_with_bicubic(x, im["outputs"]["bicubic"])}
    result = {"image_index": i, "lr_theta": lr_theta, "lr_eps": lr_eps, "steps": STEPS, "trace": trace,
              "metrics": {name: evaluate(out, im["hr"], im["y"]) for name, out in outputs.items()}}
    buffer = io.BytesIO()
    np.savez_compressed(buffer, result_json=json.dumps(result), **outputs)
    atomic_write_bytes(path, buffer.getvalue())
    return result, outputs


runs = {}
for lr_theta, lr_eps in CONFIGS:
    opt_theta, opt_eps, joint_step = make_joint_step(lr_theta, lr_eps)
    for i in DEV_IMAGES:
        runs[(lr_theta, lr_eps, i)] = run_iadrift(i, lr_theta, lr_eps, joint_step, opt_theta, opt_eps)
        m = runs[(lr_theta, lr_eps, i)][0]["metrics"]
        print(f"θ lr {lr_theta:g}, ε lr {lr_eps:g}, image {i}: PSNR IADrift {m['IADrift']['psnr']:.2f}, "
              f"+BP {m['IADrift + BP']['psnr']:.2f}, +fusion {m['IADrift + fusion']['psnr']:.2f} dB "
              f"(CSGM {images[i]['metrics']['CSGM']['psnr']:.2f}, bicubic {images[i]['metrics']['bicubic']['psnr']:.2f})")

## Results: metrics table (mean over the 4 development images)

- **PSNR** and **Δ vs bicubic** are in dB.
- **Beats bicubic** counts the images where the method's PSNR is higher than bicubic's.
- **Band SNR (dB)**: higher is better. 0 dB = band left empty, negative = added detail worse than none. Low and mid are inside the LR measurement; high and very high must be invented.
- **LR loss** after back-projection should be about 0, which checks the back-projection.

In [ ]:
def method_rows():
    """(row label, per-image metric dicts) for baselines, then every IADrift setting and output."""
    rows = [(name, [images[i]["metrics"][name] for i in DEV_IMAGES])
            for name in ("bicubic", "CSGM", "CSGM + BP", "CSGM + fusion")]
    for lr_theta, lr_eps in CONFIGS:
        for out in ("IADrift", "IADrift + BP", "IADrift + fusion"):
            rows.append((f"{out} (θ lr {lr_theta:g}, ε lr {lr_eps:g})",
                         [runs[(lr_theta, lr_eps, i)][0]["metrics"][out] for i in DEV_IMAGES]))
    return rows


bicubic_psnr = np.array([images[i]["metrics"]["bicubic"]["psnr"] for i in DEV_IMAGES])
lines = ["| Method | PSNR | Δ vs bicubic | Beats bicubic | LR loss | " + " | ".join(f"SNR {b}" for b in BAND_NAMES) + " |",
         "|---|---:|---:|---:|---:|" + "---:|" * len(BAND_NAMES)]
for name, ms in method_rows():
    p = np.array([m["psnr"] for m in ms])
    lines.append(f"| {name} | {p.mean():.2f} | {np.mean(p - bicubic_psnr):+.2f} | {int(np.sum(p > bicubic_psnr))}/{len(p)} | "
                 f"{np.mean([m['lr_loss'] for m in ms]):.2f} | "
                 + " | ".join(f"{np.mean([m['band_snr_db'][b] for m in ms]):.2f}" for b in BAND_NAMES) + " |")
display(Markdown("\n".join(lines)))

lines = ["| Method | " + " | ".join(f"{i} ({class_name(images[i]['label'])})" for i in DEV_IMAGES) + " |",
         "|---|" + "---:|" * len(DEV_IMAGES)]
for name, ms in method_rows():
    lines.append(f"| {name} | " + " | ".join(f"{m['psnr']:.2f}" for m in ms) + " |")
display(Markdown("**HR PSNR per image (dB)**\n\n" + "\n".join(lines)))

## Results: how quality changes during IADrift

Mean over the 4 images, every 50 steps, for each setting. Step 0 is the CSGM solution (evaluated with the compiled path). Dashed lines: bicubic and CSGM (eager). From left to right: HR PSNR, then band SNR for each band.

In [ ]:
panels = [("psnr", "HR PSNR (dB)")] + [(b, f"SNR {b} (dB)") for b in BAND_NAMES]
fig, axes = plt.subplots(1, len(panels), figsize=(4.2 * len(panels), 3.8))
for (key, title), ax in zip(panels, axes):
    for k, (lr_theta, lr_eps) in enumerate(CONFIGS):
        traces = [runs[(lr_theta, lr_eps, i)][0]["trace"] for i in DEV_IMAGES]
        ax.plot(traces[0]["step"], np.mean([t[key] for t in traces], axis=0),
                color=f"C{k // 2}", linestyle="-" if lr_eps else ":", label=f"θ {lr_theta:g}, ε {lr_eps:g}")
    for base, color in (("bicubic", "k"), ("CSGM", "gray")):
        value = np.mean([images[i]["metrics"][base]["psnr"] if key == "psnr"
                         else images[i]["metrics"][base]["band_snr_db"][key] for i in DEV_IMAGES])
        ax.axhline(value, color=color, linestyle="--", linewidth=1, label=base)
    ax.set(title=title, xlabel="IADrift step")
    ax.grid(alpha=0.3)
axes[0].legend(fontsize=7)
fig.tight_layout()
plt.show()

## Results: images

For each development image:
- **Top row:** original, bicubic, CSGM, CSGM + BP.
- **Bottom row:** IADrift with the previous setting (θ lr 1e-5, ε lr 1e-3), then the setting with the highest mean IADrift PSNR in the table, that setting + BP, and that setting + fusion.

Titles give HR PSNR and band SNR for low/mid/high/very high (dB).

In [ ]:
def mean_psnr(cfg, out="IADrift"):
    return np.mean([runs[(*cfg, i)][0]["metrics"][out]["psnr"] for i in DEV_IMAGES])

best_cfg = max(CONFIGS, key=mean_psnr)
old_cfg = (1e-5, 1e-3)
print(f"Highest mean IADrift PSNR on these 4 development images: θ lr {best_cfg[0]:g}, ε lr {best_cfg[1]:g}")


def show(ax, x, title, m):
    snr = "/".join(f"{m['band_snr_db'][b]:.1f}" for b in BAND_NAMES)
    ax.imshow((np.clip(x, -1, 1) + 1) / 2)
    ax.set_title(f"{title}\nPSNR {m['psnr']:.2f} dB | SNR {snr}", fontsize=8)
    ax.axis("off")


for i in DEV_IMAGES:
    im = images[i]
    fig, axes = plt.subplots(2, 4, figsize=(17, 9.4))
    axes[0, 0].imshow((im["hr"] + 1) / 2)
    axes[0, 0].set_title(f"Original HR, development image {i}\n{class_name(im['label'])}", fontsize=9)
    axes[0, 0].axis("off")
    for ax, name in zip(axes[0, 1:], ("bicubic", "CSGM", "CSGM + BP")):
        show(ax, im["outputs"][name], name, im["metrics"][name])
    panels = [(old_cfg, "IADrift", "IADrift, previous setting"),
              (best_cfg, "IADrift", "IADrift, best setting"),
              (best_cfg, "IADrift + BP", "IADrift best + BP"),
              (best_cfg, "IADrift + fusion", "IADrift best + fusion")]
    for ax, (cfg, out, title) in zip(axes[1], panels):
        result, outputs = runs[(*cfg, i)]
        show(ax, outputs[out], f"{title} (θ {cfg[0]:g}, ε {cfg[1]:g})", result["metrics"][out])
    fig.subplots_adjust(left=0.01, right=0.99, bottom=0.01, top=0.93, wspace=0.04, hspace=0.18)
    plt.show()

## Conclusions (4 development images, correct label; small sample)

**1. Gentler adaptation doesn't fix PSNR; it only slows the trade-off.**
- Every setting gives a small, brief PSNR gain over CSGM, then declines.
- The best mean gains:

  | θ lr | Peak gain over CSGM | Peak step |
  |---|---|---|
  | 1e-5 | +0.30 dB | 50 |
  | 3e-6 | +0.10 dB | 50 |
  | 1e-6 | +0.05 dB | 100–150 |

- After 1,000 steps, every setting is below CSGM (19.9 → 18.3 dB, with larger θ lr worse).
- The band curves show why: as the adaptation proceeds, low/mid SNR rise steadily (low 21 → 29–37 dB, mid 5.4 → 9–11 dB), while high/very-high SNR fall steadily (very high −6 → −8 to −11 dB). The same weight updates do both, so changing the step size moves along the trade-off but doesn't escape it.

**2. ε hardly matters; θ drives everything.** Freezing ε (lr 0) or updating it (lr 1e-3) gives nearly identical results at every θ lr. The image adaptation comes from the generator weights.

**3. Back-projection fixes the measured band but not the invented detail.**
- BP makes the low band nearly exact (SNR about 50 dB) and the LR loss 0 for every method.
- For CSGM it adds +0.8 dB, because CSGM fits the measurement poorly.
- For IADrift it adds only +0.02 to +0.2 dB, because IADrift already fits it.
- High/very-high SNR are unchanged by BP. So below the LR limit, BP alone already gives good content, and the open problem is entirely above the limit.

**4. Frequency fusion is the only variant above bicubic, and only slightly.**
- IADrift + fusion: 25.12 dB vs bicubic 24.94 (+0.18 dB, better on 3/4 images). The more adaptation, the better (+0.18 at θ lr 1e-5 vs −0.07 at 1e-6), because its mid-band content improves.
- CSGM + fusion is below bicubic (−0.65 dB).
- This uses IADrift only between 0.0625 and 0.125 cycles/pixel, so it looks like bicubic, not like a sharp generative result. It's a PSNR-oriented variant.

**5. Caution about point 4.** CSGM + BP has a *higher* mid-band SNR (12.6 dB) than any IADrift output (≤ 11.1 dB). So part of the fusion gain may come from fitting the measurement in the mid band, which back-projection alone also achieves, rather than from the generator prior. Controls (run in notebook 04, Check 1): **bicubic + BP**, **CSGM + BP + fusion** and **IADrift + BP + fusion**.

**6. Visual quality.** The setting with the best PSNR (θ lr 1e-6) changes the CSGM image least and keeps its blocky artifacts. The fused outputs look like bicubic. The sharp IADrift detail that looked better in the earlier figures is exactly what PSNR penalizes. A perceptual metric (LPIPS) is needed to judge it fairly.

**Implications.**
- On PSNR, IADrift as a joint (θ, ε) optimization doesn't beat CSGM by more than about 0.3 dB, and only with early stopping (about 50 steps at θ lr 1e-5).
- Neither method beats bicubic without frequency fusion.
- The method's clear strength is reconstructing the band the measurement determines.

---
# Additional experiment: one image, correct label vs 5 random wrong labels

**Question.** How much does the class label change CSGM and IADrift on a single image, and does it matter more before or after the generator adapts?

**What we did.**
- **Image:** development image **8117** (streetcar), one of the 4 development images above. `check_dev` again refuses test images.
- **Labels:** the correct label plus **5 wrong labels**, drawn uniformly at random from the other 999 classes. They're fixed by seed (2031, image index), so re-running gives the same 5.
- **Per label:**
  - **CSGM:** frozen generator, Adam lr 0.02, 5,000 steps, 2 restarts, keeping the restart with the lowest LR loss. The `eps`/`noise_labels` starting points are identical for every label (same seed per restart), so only the label differs. Runs are cached in `csgm_dev_5000steps_lr0.02/`; the correct-label run is reused from before.
  - **IADrift** from that label's CSGM solution, keeping the **same label**: joint Adam on θ and ε, 1,000 steps, θ lr 1e-5, ε lr 1e-3. That's the setting with the best fusion result and the largest early PSNR peak above. It's set in `LABEL_CONFIGS`; add settings there to compare more.
  - Every 50 steps: HR PSNR, LR loss and band SNR (evaluation only).
  - Outputs, as above: CSGM, CSGM + BP, CSGM + fusion, IADrift, IADrift + BP, IADrift + fusion. Bicubic is the same for every label.
- **Metrics, as above:** HR PSNR, LR loss, band SNR, eager evaluation for final images.

**Caveats.**
- One image and 5 wrong labels: this shows how much the label *can* matter, not an average effect.
- The 100-image test benchmark (oracle vs one wrong label) is still the place for that.

Results are cached in `~/work/datasets/ImageNetV2/iadrift_dev_labels_v1/`.

In [ ]:
LABEL_IMAGE = 8117
NUM_WRONG = 5
LABEL_SEED = 2031
LABEL_CONFIGS = [(1e-5, 1e-3)]  # IADrift settings (θ lr, ε lr)
LABEL_OUT = DATA_DIR / "iadrift_dev_labels_v1"

check_dev([LABEL_IMAGE], train_indices, test_indices)  # refuses test images
lim = images[LABEL_IMAGE]
true_label = lim["label"]
rng = np.random.default_rng([LABEL_SEED, LABEL_IMAGE])
wrong_labels = [int(c) for c in rng.choice(np.setdiff1d(np.arange(1000), [true_label]), NUM_WRONG, replace=False)]
label_list = [("correct", true_label)] + [(f"wrong {k + 1}", c) for k, c in enumerate(wrong_labels)]
print("Labels:", ", ".join(f"{name}: {c} ({class_name(c)})" for name, c in label_list))


def csgm_for_label(label):
    """Cached CSGM for this image and label; returns (best restart, arrays)."""
    condition = "oracle" if label == true_label else f"label{label}"
    restarts = [csgm_cached(LABEL_IMAGE, condition, r, label, lr_all[LABEL_IMAGE], lim["hr"])
                for r in range(FROZEN_CONFIG["num_restarts"])]
    best = min(range(len(restarts)), key=lambda r: (restarts[r][0]["final_loss"], r))
    return best, restarts[best][1]


def run_iadrift_label(label, csgm_arrays, lr_theta, lr_eps, joint_step, opt_theta, opt_eps):
    """Same procedure as run_iadrift above, for an arbitrary label; cached as an .npz."""
    path = LABEL_OUT / f"img{LABEL_IMAGE:05d}_label{label}_lrt{lr_theta:g}_lre{lr_eps:g}.npz"
    if path.exists():
        with np.load(path) as data:
            return json.loads(str(data["result_json"])), {k: data[k] for k in ("IADrift", "IADrift + BP", "IADrift + fusion")}

    theta = jax.device_put(params)
    eps, noise_labels = jnp.asarray(csgm_arrays["eps"]), jnp.asarray(csgm_arrays["noise_labels"])
    state_theta, state_eps = opt_theta.init(theta), opt_eps.init(eps)
    trace = {"step": [], "psnr": [], "lr_loss": [], **{b: [] for b in BAND_NAMES}}

    def record(step):
        x = np.asarray(generate_jit(theta, eps, label, noise_labels)[0])
        m = evaluate(x, lim["hr"], lim["y"])
        trace["step"].append(step)
        trace["psnr"].append(m["psnr"])
        trace["lr_loss"].append(m["lr_loss"])
        for b in BAND_NAMES:
            trace[b].append(m["band_snr_db"][b])

    record(0)
    for step in range(1, STEPS + 1):
        theta, eps, state_theta, state_eps, _ = joint_step(
            theta, eps, state_theta, state_eps, lim["y"], label, noise_labels)
        if step % EVAL_EVERY == 0:
            record(step)

    x, _, _ = eager_metrics(theta, eps, label, noise_labels, lim["y"], lim["hr"])
    outputs = {"IADrift": x, "IADrift + BP": backproject(x, lim["y"]),
               "IADrift + fusion": fuse_with_bicubic(x, lim["outputs"]["bicubic"])}
    result = {"image_index": LABEL_IMAGE, "label": label, "lr_theta": lr_theta, "lr_eps": lr_eps, "steps": STEPS,
              "trace": trace, "metrics": {k: evaluate(v, lim["hr"], lim["y"]) for k, v in outputs.items()}}
    buffer = io.BytesIO()
    np.savez_compressed(buffer, result_json=json.dumps(result), **outputs)
    atomic_write_bytes(path, buffer.getvalue())
    return result, outputs


label_csgm = {}
for name, label in label_list:
    restart, arrays = csgm_for_label(label)
    outs = {"CSGM": arrays["x_hat"], "CSGM + BP": backproject(arrays["x_hat"], lim["y"]),
            "CSGM + fusion": fuse_with_bicubic(arrays["x_hat"], lim["outputs"]["bicubic"])}
    label_csgm[label] = {"restart": restart, "arrays": arrays, "outputs": outs,
                         "metrics": {k: evaluate(v, lim["hr"], lim["y"]) for k, v in outs.items()}}
    print(f"{name:<8} {class_name(label):<24} CSGM r{restart}: PSNR {label_csgm[label]['metrics']['CSGM']['psnr']:.2f} dB, "
          f"LR loss {label_csgm[label]['metrics']['CSGM']['lr_loss']:.1f}")

label_runs = {}
for lr_theta, lr_eps in LABEL_CONFIGS:
    opt_theta, opt_eps, joint_step = make_joint_step(lr_theta, lr_eps)
    for name, label in label_list:
        label_runs[(lr_theta, lr_eps, label)] = run_iadrift_label(
            label, label_csgm[label]["arrays"], lr_theta, lr_eps, joint_step, opt_theta, opt_eps)
        m = label_runs[(lr_theta, lr_eps, label)][0]["metrics"]
        print(f"θ lr {lr_theta:g}, ε lr {lr_eps:g}, {name:<8} {class_name(label):<24} IADrift {m['IADrift']['psnr']:.2f}, "
              f"+BP {m['IADrift + BP']['psnr']:.2f}, +fusion {m['IADrift + fusion']['psnr']:.2f} dB")

### Results: metrics per label (image 8117)

- **Δ vs bicubic** is in dB.
- **Band SNR** is in dB as above: low/mid are inside the LR measurement, high/very high are invented.
- The second table compares the correct label with the **mean ± standard deviation over the 5 wrong labels**, and gives how many wrong labels beat the correct one on PSNR.

In [ ]:
LABEL_METHODS = ["CSGM", "CSGM + BP", "CSGM + fusion"] + [
    f"{out} (θ lr {lt:g}, ε lr {le:g})" for lt, le in LABEL_CONFIGS
    for out in ("IADrift", "IADrift + BP", "IADrift + fusion")]


def label_metric(label, method):
    if method.startswith("CSGM"):
        return label_csgm[label]["metrics"][method]
    out, cfg = method.split(" (θ lr ")
    lt, le = (float(v) for v in cfg.rstrip(")").split(", ε lr "))
    return label_runs[(lt, le, label)][0]["metrics"][out]


bic = lim["metrics"]["bicubic"]
head = "| Label | Method | PSNR | Δ vs bicubic | LR loss | " + " | ".join(f"SNR {b}" for b in BAND_NAMES) + " |"
lines = [head, "|---|---|---:|---:|---:|" + "---:|" * len(BAND_NAMES),
         f"| (any) | bicubic | {bic['psnr']:.2f} | +0.00 | {bic['lr_loss']:.2f} | "
         + " | ".join(f"{bic['band_snr_db'][b]:.2f}" for b in BAND_NAMES) + " |"]
for name, label in label_list:
    for method in LABEL_METHODS:
        m = label_metric(label, method)
        lines.append(f"| {name}: {class_name(label)} | {method} | {m['psnr']:.2f} | {m['psnr'] - bic['psnr']:+.2f} | "
                     f"{m['lr_loss']:.2f} | " + " | ".join(f"{m['band_snr_db'][b]:.2f}" for b in BAND_NAMES) + " |")
display(Markdown("\n".join(lines)))

wrong = [c for name, c in label_list if name != "correct"]
lines = ["| Method | Metric | Correct label | Wrong labels: mean ± sd | Wrong beats correct |", "|---|---|---:|---:|---:|"]
for method in LABEL_METHODS:
    for key, title in [("psnr", "PSNR"), ("lr_loss", "LR loss")] + [(b, f"SNR {b}") for b in BAND_NAMES]:
        def get(c):
            m = label_metric(c, method)
            return m["band_snr_db"][key] if key in BAND_NAMES else m[key]
        w = np.array([get(c) for c in wrong])
        beats = f"{int(np.sum(w > get(true_label)))}/{len(w)}" if key == "psnr" else ""
        lines.append(f"| {method} | {title} | {get(true_label):.2f} | {w.mean():.2f} ± {w.std(ddof=1):.2f} | {beats} |")
display(Markdown("**Correct label vs the 5 wrong labels**\n\n" + "\n".join(lines)))

### Results: IADrift trajectories per label

Every 50 steps, for each label: HR PSNR, then band SNR per band. Step 0 is that label's CSGM solution. Black = correct label; dashed gray = bicubic.

In [ ]:
label_panels = [("psnr", "HR PSNR (dB)")] + [(b, f"SNR {b} (dB)") for b in BAND_NAMES]
for lt, le in LABEL_CONFIGS:
    fig, axes = plt.subplots(1, len(label_panels), figsize=(4.2 * len(label_panels), 3.8))
    for (key, title), ax in zip(label_panels, axes):
        for k, (name, label) in enumerate(label_list):
            t = label_runs[(lt, le, label)][0]["trace"]
            ax.plot(t["step"], t[key], color="k" if name == "correct" else f"C{k}",
                    linewidth=2 if name == "correct" else 1, label=f"{name}: {class_name(label)}")
        value = bic["psnr"] if key == "psnr" else bic["band_snr_db"][key]
        ax.axhline(value, color="gray", linestyle="--", linewidth=1, label="bicubic")
        ax.set(title=title, xlabel="IADrift step")
        ax.grid(alpha=0.3)
    axes[0].legend(fontsize=7)
    fig.suptitle(f"Development image {LABEL_IMAGE}: IADrift (θ lr {lt:g}, ε lr {le:g}) per label", fontsize=11)
    fig.tight_layout()
    plt.show()

### Results: images per label

- **First row:** original and bicubic.
- **Then one row per label:** CSGM, CSGM + BP, IADrift, IADrift + BP, IADrift + fusion, for the first setting in `LABEL_CONFIGS`.

Titles give HR PSNR and band SNR for low/mid/high/very high (dB).

In [ ]:
lt, le = LABEL_CONFIGS[0]
cols = ["CSGM", "CSGM + BP", "IADrift", "IADrift + BP", "IADrift + fusion"]
fig, axes = plt.subplots(len(label_list) + 1, len(cols), figsize=(3.6 * len(cols), 3.9 * (len(label_list) + 1)))
for ax in axes[0]:
    ax.axis("off")
axes[0, 0].imshow((lim["hr"] + 1) / 2)
axes[0, 0].set_title(f"Original HR, development image {LABEL_IMAGE}\n{class_name(true_label)}", fontsize=9)
show(axes[0, 1], lim["outputs"]["bicubic"], "bicubic", bic)
for row, (name, label) in enumerate(label_list, start=1):
    result, outs = label_runs[(lt, le, label)]
    for ax, col in zip(axes[row], cols):
        if col.startswith("CSGM"):
            x, m = label_csgm[label]["outputs"][col], label_csgm[label]["metrics"][col]
        else:
            x, m = outs[col], result["metrics"][col]
        show(ax, x, f"{col}, {name}: {class_name(label)}", m)
fig.subplots_adjust(left=0.01, right=0.99, bottom=0.01, top=0.97, wspace=0.05, hspace=0.3)
plt.show()